# 使用 AgentKit 构建、部署和优化智能体工作流

## 简介

我们在 DevDay 2025 上发布了 [AgentKit](https://openai.com/index/introducing-agentkit/)，这是一整套面向开发者和企业的工具，用于构建、部署和优化智能体。AgentKit 由一组相互关联的构建模块组成：

* [Agent Builder](https://platform.openai.com/docs/guides/agents/agent-builder)：以可视化方式构建和迭代智能体工作流  
* [ChatKit](https://platform.openai.com/docs/guides/chatkit)：轻松将基于对话的工作流嵌入你的应用  
* [Evals](https://platform.openai.com/docs/guides/evals?api-mode=responses)：提升基于 LLM 的应用的性能

**本 cookbook 将带你端到端地体验 AgentKit——我们将构建、部署并优化一个应用。你将了解 AgentKit 的各个构建模块如何相互衔接，从而帮助你更快、更可靠地将智能体工作流投入生产。**

我们将依次完成以下步骤：

1. 在 Agent Builder 中构建一个工作流，作为应用的后端  
2. 使用 ChatKit Web 组件部署一个前端聊天应用  
3. 在 Evals 中通过提示词优化和追踪评分（trace grading）优化工作流性能

## 使用 Agent Builder 构建多智能体工作流

我们首先使用 Agent Builder 创建支撑应用的初始工作流。Agent Builder 是一个可视化画布，让你通过拖放节点来设计智能体工作流。你可以在[这里](https://platform.openai.com/docs/guides/agent-builder)进一步了解 Agent Builder，包括更多功能以及受支持节点的完整列表；不过在本 cookbook 中，我们将创建一个由三个按顺序工作的 Agent 节点组成的简单工作流。

我们将构建一个简单的应用，通过精选的学习推荐帮助人们加速职业发展。用户可以上传简历并告诉我们他们的理想工作，然后会根据需要提升的技能收到一组精选的学习课程。因此，我们将创建三个智能体：

1. **简历提取**智能体：解析上传的简历，提取相关的技能和经历  
2. **职业分析**智能体：分析相对于目标工作所存在的知识差距  
3. **课程推荐**智能体：利用上游信息推荐相关的在线课程。

下面我们依次构建这些智能体。

<div><center><img src="https://cdn.openai.com/cookbook/agent_walkthrough/agentkit_walkthrough_workflow.png" /></center></div>

### 1. 简历提取智能体

该智能体负责解析上传的简历，并以结构化输出返回技能和经历，供下游分析使用。我们将使用以下提示词：


```text
Extract and summarize information from the input resume, organizing your output by category and providing context where available.
- Analyze the provided input to identify skills and professional experiences.
- For each skill or experience, extract the supporting context or evidence from the text (e.g., for the skill of Python, context might be “used Python in data analysis for three years at [Company]”).
- Continue reviewing the text until all skills and experiences are extracted.
```

该智能体使用 `gpt-5`，初始推理强度为 `minimal`，之后如有需要随时可以更换模型。此外，我们会强制使用结构化响应（将 Output format 设为 JSON 并添加 schema），指示模型严格按照我们需要的数据结构返回。（该结构化输出的 JSON schema 可以在[这里](https://cdn.openai.com/cookbook/agent_walkthrough/Skills_schema.json)找到。）

<div><center><img src="https://cdn.openai.com/cookbook/agent_walkthrough/agentkit_walkthrough_json_schema.png" width="50%"/></center></div>

### 2. 职业分析智能体

该智能体会分析一个人为了实现期望的职业目标所存在的技能和知识差距。该智能体使用 `gpt-5`，推理强度选择 `low`，这应能为这一层级的分析提供足够的推理，同时保持足够快的响应速度。

```text
Your role is to analyze skill and knowledge gaps for an individual to progress to a desired professional or career goal. 

You will receive a list of the already-obtained skills and experiences of an individual, as well as a description of the goal. First, understand the goal and analyze the critical skills or knowledge areas required for achieving the goal. Then, compare the requirements to what the individual already possesses. 

Return a list of the top 3-5 skills that the individual does not possess, but are important for their professional goal. Along with each skill, include a brief description.

Individual's expressed goal: 
{{workflow.input_as_text}}

Already-obtained skills and experiences:
{{input.output_text}}
```

注意，我们的提示词中包含了来自前面节点、用 {{brackets}} 括起来的上下文。你也可以点击 "Add context" 查看提供给模型的上下文变量。

### 3. 课程推荐智能体

该智能体将使用网页搜索工具查找并挑选与已识别技能差距相匹配的在线培训课程。我们使用推理强度为 `minimal` 的 `gpt-5`，并为该智能体配备 Web Search。

```text
Your job is to identify and recommend online training courses that help develop one or more of the skills identified. Given the list of required skills and descriptions below, return a list of 3-5 online courses along with course details.

Skills: {{input.output_text}}
```

## 测试工作流

**初始工作流构建完成后，我们可以使用 Agent Builder 中的 Preview 功能试运行一下！** 我们先发布（Publish）工作流，这会创建一个带名称和版本号的副本（具有唯一的工作流 ID），可以与同事共享，也可以根据需要部署或在不同版本之间回退。

Preview 让你可以直接在 Agent Builder 中，像聊天用户一样与工作流交互。如果我们上传一份简历，输入对理想工作的描述并点击 Submit，就能在屏幕左侧看到工作流逐个节点地推进，并在右侧看到每个节点的输出。

例如，我上传了一位希望成为学区督学的教师的简历。


<div><center><img src="https://cdn.openai.com/cookbook/agent_walkthrough/agentkit_walkthrough_workflow_preview.png" /></center></div>

我们可以跟随工作流依次完成简历解析、技能分析和网页搜索。工作流完成后，正如预期，我们看到了一份符合搜索条件的在线项目列表。

**一切看起来都很好——现在可以部署应用了！** 请确保发布了所有改动，并获取工作流 ID。我们可以选择屏幕顶部的 "Code" 再次查看该 ID。

<div><center><img src="https://cdn.openai.com/cookbook/agent_walkthrough/agentkit_walkthrough_code.png" /></center></div>

注意，你可以通过 "Agents SDK" 选项卡获取支撑你刚构建的工作流的代码，这些代码使用 Agents SDK 包实现（[JavaScript/TypeScript](https://github.com/openai/openai-agents-js) 或 [Python](https://github.com/openai/openai-agents-python)）。如果你想在自己的环境中运行工作流，或者通过自定义功能进一步开发它，这是一个很好的选择。（不过，这样你就会错失以集成方式使用 AgentKit 带来的一些好处，我们将在下文介绍。）

## 使用 ChatKit 部署聊天应用

为了部署应用，我们将使用 [ChatKit 入门模板](https://github.com/openai/openai-chatkit-starter-app)，借助 ChatKit Web 组件快速搭建一个基于对话的应用。

在此之前，有必要介绍一下 AgentKit 整套工具为部署智能体工作流提供的全部选项。我们已经看到，你可以在 Agent Builder 中构建工作流，然后直接在工具内运行（Preview），或者将其导出为 Agents SDK 代码在你自己的环境中使用。现在，我们将演示如何使用 Agent Builder 的工作流 ID，在你自己的前端中嵌入聊天体验，并以你创建的工作流作为后端。（顺便一提，你也可以只使用 ChatKit SDK 提供的丰富聊天 GUI，而不使用工作流后端——更多信息请参阅[这里](https://platform.openai.com/docs/guides/custom-chatkit)。）

现在就用 ChatKit 入门模板开始，并接入我们的工作流。借助入门模板，使用新创建的工作流搭建对话式应用非常简单。只需按照仓库中的 [Getting Started](https://github.com/openai/openai-chatkit-starter-app?tab=readme-ov-file#getting-started) 说明操作：在 `.env.local` 中将 Agent Builder 的工作流 ID 填为 `NEXT_PUBLIC_CHATKIT_WORKFLOW_ID` 的值，然后运行 `npm install`，再运行 `npm run dev` 来测试应用。

只需几分钟，工作流就被嵌入到一个已经运行起来的前端聊天界面中！

<div><center><img src="https://cdn.openai.com/cookbook/agent_walkthrough/agentkit_walkthrough_chat_app.png" /></center></div>

## 快速迭代工作流和用户体验

AgentKit 最有价值的方面之一，就是它能让你非常快速地试验、迭代和改进智能体应用。下面我们做一些快速改动，以改进应用的功能并提供更丰富的聊天体验。

**首先，添加一些自定义主题**，为前端增添风格，同时保留原生的聊天体验。这里一个很好的资源是 [ChatKit Studio](https://chatkit.studio/)，其中包括一个用于探索 ChatKit 自定义选项的 playground、一个 Widget Builder（我们稍后就会看到它的实际效果），以及可供参考的示例应用和展示库。为了获得自定义主题，我们将使用 ChatKit [Playground](https://chatkit.studio/playground) 以可视化方式选择所需的样式选项，然后点击 Playground 屏幕顶部的 `</>` 图标获取配置代码。

我们会用这段代码中的 `theme` 对象覆盖位于 [lib/config.ts](https://github.com/openai/openai-chatkit-starter-app/blob/main/lib/config.ts) 中的默认主题。既然打开了这个文件，我们还会把起始提示、问候语和占位文本调整为更合适的值：

```text
export const GREETING = "Upload your resume, and tell me the job you're looking to get!";
export const PLACEHOLDER_INPUT = "Describe your dream job, and don't forget to attach your resume!"; 
```

**接下来，我们设计一个自定义 widget**，以更直观的格式展示推荐课程，让用户更容易理解和比较。我们可以使用 [Widget Builder](https://widgets.chatkit.studio/)，只需描述想要的输出，就能得到一个由 LLM 生成的起点，再进一步编辑。在本示例中，我们将以简洁、结构化的列表形式展示课程，并在推荐下方显示一段摘要。在 Widget Builder 中，我们不仅可以看到 widget 代码（左上），还能看到示例数据（左下），以及这些数据在 widget 中如何呈现给最终用户（右侧）。

<div><center><img src="https://cdn.openai.com/cookbook/agent_walkthrough/agentkit_walkthrough_widget_builder.png" /></center></div>

对设计满意后，我们可以下载 .widget 文件。（下方截图中使用的文件位于[这里](https://cdn.openai.com/cookbook/agent_walkthrough/Course%20recommendation.widget)。）

要在聊天应用中真正使用我们设计的自定义 widget，**我们需要指示工作流返回该 widget 组件**，作为推荐智能体响应的一部分。因此，我们回到 `Course recommendations` 智能体，将 Output format 选为 `Widget`，并上传 .widget 文件。

智能体会自动知道要输出填充 widget 所需的 JSON 格式。不过，我们需要更新智能体的提示词，更精确地告诉模型需要获取每门课程的哪些信息。

```text
Your job is to identify and recommend online training courses that help develop one or more of the skills identified. Given the list of required skills, return a list of 3-5 online courses along with course details including course name, provider (school or program), recommendation reason (a brief sentence on why you're recommending the course), course format, and URL. In addition to the list of courses, share a few-sentence summary of the recommendations you're making. 
```

最后，由于我们处理的是简历，我们会在工作流中添加一个护栏（guardrail），确保不会把任何个人身份信息（PII）传播到不该出现的地方。我们会把这个护栏插入到简历解析智能体和职业分析智能体之间，这有助于防止简历智能体下游的任何环节接触到姓名或联系方式等 PII。


<div><center><img src="https://cdn.openai.com/cookbook/agent_walkthrough/agentkit_walkthrough_guardrails.png" /></center></div>

## 使用提示词优化和追踪评分提升系统性能

现在我们来看看与 Evals 的原生集成如何让优化单个智能体和整个工作流都变得轻松。假设我们的聊天应用已经部署——也许面向一批初始的内部用户或 beta 测试者——并且我们已经有了一些用户与系统交互的真实示例。

如果要将这个应用开发成对性能和质量要求极高的生产级系统，我们会希望更早、更系统地将评估纳入开发流程。（你可以在 [Eval Driven System Design](https://cookbook.openai.com/examples/partners/eval_driven_system_design/receipt_inspection) 中进一步了解如何构建一组初始评估并建立真实标准、将评估映射到业务指标，以及逐步改进系统以实现业务目标。）

但在本 cookbook 中，我们不会过多关注 LLM 应用开发中评估背后的技术，而是更关注 AgentKit 如何让你更轻松地实施这些技术。我们将通过两种方式提升性能：首先使用提示词优化工具**优化工作流中的单个智能体节点**，然后使用追踪评分**优化整个工作流**。

### 单个智能体优化

我们希望深入研究 Course recommendations 智能体，看看能否提升它给用户的推荐质量。我们已经从测试环境中整理出该智能体的一些示例提示词。（你也可以在 API 平台的 [Logs 选项卡](https://platform.openai.com/logs?api=responses)中查看已完成的请求。对于本 cookbook 示例，你可以在[这里](https://cdn.openai.com/cookbook/agent_walkthrough/course_recommendations_dataset.csv)获取我们将使用的数据集。）

我们可以直接从 Agent Builder 开始优化智能体。选择 Course recommendations 智能体，然后点击智能体弹窗右下角的 "Evaluate"。这会直接带我们进入 Evals 中的 **Datasets** 功能。

可以看到智能体的配置已经被复制过来，我们可以开始优化了。首先上传包含示例提示词的数据文件（注意列名应与你的输入和输出变量相匹配），然后点击 "Generate output" 生成响应。

<div><center><img src="https://cdn.openai.com/cookbook/agent_walkthrough/agentkit_walkthrough_generate_output.png" /></center></div>

现在，我们来创建一些**人工标注**和**模型评分器**。我们选择 "Columns"，添加 Rating（点赞/点踩）和 Feedback（文本输入）两列，然后人工审查样本，为这些字段填写一些高质量的反馈。我们还会添加几个模型评分器，它们会根据我们指定的标准自动评估智能体的输出。在本示例中，我们可能会关心：课程推荐是否与识别出的技能差距相关（相关性）、是否覆盖了所有技能差距（覆盖度），以及给出的推荐摘要是否恰当（风格）。

以下是针对每项标准的模型评分器提示词示例：

```text
[relevance] You are evaluating whether a list of recommended courses is relevant to the skills described. Return a pass if all courses are relevant to at least one skill, and fail otherwise.

[coverage] You are evaluating whether a list of recommended courses covers all of the skills described. Return a pass if all of the skills are covered by at least one course, and fail otherwise.

[summary] You are evaluating whether the summary recommendation provided is relevant, thoughtful, and related to the recommended courses proposed. Evaluate the summary recommendation on a scale of 0 to 1, with 1 being the highest quality. 
```

我们的模型评分器使用 GPT-5，并为摘要评分器设置 0.7 的阈值。

现在选择 Grade > All graders，针对系统输出运行这些评分器。随着评分的进行，我们会看到单元格陆续填充，显示每个示例在模型评分器各项标准上的得分。

<div><center><img src="https://cdn.openai.com/cookbook/agent_walkthrough/agentkit_walkthrough_grading.png" /></center></div>

**现在，见证奇迹的时刻到了：我们可以点击 Optimize，根据我们提供的反馈（包括模型评分器的输出和人工提供的反馈）自动重写提示词。**

查看新提示词会发现，其中新增了 **Requirements** 和 **Output** 格式两个部分，指示模型让课程描述更加具体，并力求更好地覆盖不同的技能。

```text
Requirements:

- Use the web search tool to find and verify real, currently available online courses and their direct URLs.
- Return 3–5 courses that collectively cover the skills. If a course spans multiple skills, indicate it. 
- Be specific and concise. Each course description must be one sentence (max 35 words) focused on outcomes and topics tied to the skills. 
- Provide plain text only; no citations or references.

Output format:

1) Summary (2–4 sentences) explaining how the selections address the skills and any coverage tradeoffs. 
2) Courses (3–5 items). For each course, include exactly:` 
   - Course name — Provider (school or program)
   - Description: [one sentence, max 35 words]  
   - URL: [direct course page]
   - Skills covered: [list skill names from below]
```

现在，我们可以点击 Update，将新提示词自动插入 Agent Builder 中的工作流。**只用了几分钟，我们就利用真实的提示词示例和反馈自动提升了系统性能——而且全部是在 Agent Builder 和 Evals 工具中直接完成的。**

（虽然在本 cookbook 中我们使用评分输出自动优化了提示词，但查看具体的失败示例、了解模型可能在哪些方面出错，往往也非常有帮助。这种分析可以帮助我们生成更精确的人工反馈或模型评分反馈，甚至合成数据来提升针对特定失败模式的表现。）


### 整个工作流优化

当我们对每个单独智能体节点的性能感到满意后，就可以把注意力转向整个工作流。

以前，要分析和改进涉及多个智能体的复杂工作流的性能，你需要通读工作流的完整追踪，才能弄清系统究竟在哪里、为什么失败。这可能非常耗时，尤其是在追踪示例数量很多的情况下。

借助**追踪评分**，我们现在可以使用自动化的模型评分，对整组追踪进行端到端评估。我们会创建评分器来描述想要纠正的行为，并在整个数据集上自动运行这些评分器。

首先，我们回到 Agent Builder 中的工作流，点击屏幕顶部的 Evaluate。这让我们可以深入查看工作流运行所生成的追踪，包括检查所有节点的输入和输出（在本例中，即每个智能体的提示词和响应）。

我们可以创建在整个模型追踪上运行的评分器，为**跨多个智能体**的端到端工作流定义评估标准。例如，我们可能希望确保最终的推荐摘要（智能体 3 的输出）与用户最初关于职业目标的输入（智能体 1 的输入）相关。此外，我们可能还想检查推荐的课程（智能体 3 的输出）不会与用户已具备的技能（智能体 1 的输出）重复。

<div><center><img src="https://cdn.openai.com/cookbook/agent_walkthrough/agentkit_walkthrough_traces.png" /></center></div>

如果你的工作流包含条件语句或 while 循环，还可以针对更复杂的多步行为进行评分，例如支持智能体在未升级给主管的情况下，与用户的交互不应超过三轮回复。

有了一整套评分标准后，我们选择 Grade all 为追踪评分。此操作会将我们带到 Evaluations 选项卡，在那里可以看到已经创建了一个新的评估，并启动了一次评估运行。

<div><center><img src="https://cdn.openai.com/cookbook/agent_walkthrough/agentkit_walkthrough_trace_evals.png" /></center></div>

然后，我们可以深入查看失败用例的工作流追踪，更好地理解特定工作流运行失败的原因，以及如何改进系统以避免失败。通过迭代地识别失败模式、评估系统性能并有针对性地进行改进，这种方法可以帮助你更高效地优化复杂的工作流。

## 回顾与资源

我们演示了 **Agent Builder**、**ChatKit** 和 **Evals** 如何协同工作，帮助你构建、部署和优化智能体工作流。

通过一个具体示例——一个分析简历、识别技能差距并推荐在线课程的职业发展应用——我们看到：Agent Builder 让设计和构建多智能体工作流变得简单，ChatKit 让我们能将这些工作流嵌入丰富且可定制的聊天 UI，而 Evals 则通过基于真实数据的提示词优化和追踪评分形成闭环。

想了解更多，以下是本 cookbook 中提到的部分资源：

* [Agent Builder 文档](#)
* [ChatKit 入门模板](#)
* [ChatKit Studio](#)
* [Agents SDK](#)
* [Evals](#)

祝你构建愉快！